In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure, morphology

def count_scales(image_path, hsv_lower, hsv_upper):
    """
    Count individual scales using image processing techniques
    
    Parameters:
    - image_path: Path to the butterfly wing image
    - hsv_lower: Lower HSV boundary
    - hsv_upper: Upper HSV boundary
    
    Returns:
    - Number of distinct scales
    - Visualization of detected scales
    """
    # Read the image
    image = cv2.imread(image_path)
    
    # Convert to HSV
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Create color mask
    color_mask = cv2.inRange(hsv_image, hsv_lower, hsv_upper)
    
    # Morphological operations to clean up the mask
    kernel = np.ones((3,3), np.uint8)
    cleaned_mask = cv2.morphologyEx(color_mask, cv2.MORPH_OPEN, kernel)
    cleaned_mask = cv2.morphologyEx(cleaned_mask, cv2.MORPH_CLOSE, kernel)
    
    # Label connected regions (potential scales)
    labeled_mask = measure.label(cleaned_mask)
    
    # Filter scales based on size
    scale_min_size = 10  # Minimum pixel area to be considered a scale
    scale_max_size = 500  # Maximum pixel area to be considered a scale
    
    # Collect properties of potential scales
    props = measure.regionprops(labeled_mask)
    
    # Filter scales based on size and other properties
    valid_scales = [
        prop for prop in props 
        if scale_min_size < prop.area < scale_max_size
    ]
    
    # Visualize results
    plt.figure(figsize=(15,5))
    
    # Original image
    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title('Original Image')
    plt.axis('off')
    
    # Color mask
    plt.subplot(1, 3, 2)
    plt.imshow(cleaned_mask, cmap='gray')
    plt.title('Color Mask')
    plt.axis('off')
    
    # Scales visualization
    overlay = image.copy()
    for scale in valid_scales:
        # Draw a small rectangle for each scale
        minr, minc, maxr, maxc = scale.bbox
        cv2.rectangle(overlay, (minc, minr), (maxc, maxr), (0, 255, 0), 1)
    
    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title(f'Detected Scales: {len(valid_scales)}')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return len(valid_scales)

def detect_orange_scales(image_path):
    """
    Automatically detect potential orange scales
    """
    # Multiple potential orange ranges
    orange_ranges = [
        {
            'lower': np.array([0, 50, 50]),    # Broad lower range
            'upper': np.array([30, 255, 255]), # Broad upper range
            'name': 'Broad Orange Range'
        },
        {
            'lower': np.array([10, 100, 100]), # Medium range
            'upper': np.array([20, 255, 255]), 
            'name': 'Medium Orange Range'
        },
        {
            'lower': np.array([10, 100, 100]), # Mid-Narrow, intense range
            'upper': np.array([25, 255, 255]),
            'name': 'Narrow Intense Orange Range'
        },
        {
            'lower': np.array([15, 150, 150]), # Narrow, intense range
            'upper': np.array([25, 255, 255]),
            'name': 'Narrow Intense Orange Range'
        }
    ]
    
    # Detect scales for each range
    scale_detections = []
    
    for range_config in orange_ranges:
        num_scales = count_scales(
            image_path, 
            range_config['lower'], 
            range_config['upper']
        )
        
        scale_detections.append({
            'range': range_config,
            'num_scales': num_scales
        })
    
    # Display detection results
    print("\nDetected Scales for Different Ranges:")
    for detection in scale_detections:
        print(f"{detection['range']['name']}:")
        print(f"  Lower HSV: {detection['range']['lower']}")
        print(f"  Upper HSV: {detection['range']['upper']}")
        print(f"  Number of Scales: {detection['num_scales']}\n")
    
    # Interactive selection
    # while True:
    #     try:
    #         choice = input("Select the range number (1-4): ")
    #         choice = int(choice)
    #         if 1 <= choice <= 4:
    #             selected = scale_detections[choice-1]
                
    #             confirm = input(f"Confirm {selected['range']['name']} with {selected['num_scales']} scales? (y/n): ").lower()
    #             if confirm == 'y':
    #                 return selected['range']['lower'], selected['range']['upper']
    #         else:
    #             print("Invalid choice. Please enter 1, 2, or 3.")
    #     except ValueError:
    #         print("Please enter a valid number.")

# Example usage
image_path = '/Users/fseixas/Library/CloudStorage/Dropbox/seixas-rustyscales/wing.photos/CWvsWC/4WC01f.jpg'
lower, upper = detect_orange_scales(image_path)
